# InferLite on free Google Colab GPUs

This notebook runs **measured** inference experiments (TTFT, tokens/sec, P50/P95/P99, memory, load time) and **labels unsupported methods instead of inventing scores**.

**Setup:** Runtime → Change runtime type → T4 GPU.

Upload or clone the InferLite repo so `REPO_ROOT` points at the project root (the folder that contains `backend/` and `configs/`).

In [3]:
import os, sys, pathlib, subprocess

def is_repo(path):
    return (pathlib.Path(path) / "backend" / "research").is_dir()

REPO_ROOT = None
for cand in [
    pathlib.Path("/content/llm-inferlite"),
    pathlib.Path("/content/llm-inferlite-main"),
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent,
    pathlib.Path("/content"),
]:
    if is_repo(cand):
        REPO_ROOT = cand
        break
    for name in ("llm-inferlite", "llm-inferlite-main"):
        nested = cand / name
        if is_repo(nested):
            REPO_ROOT = nested
            break
    if REPO_ROOT is not None:
        break

if REPO_ROOT is None:
    dest = pathlib.Path("/content/llm-inferlite")
    print("Repo not found. Cloning https://github.com/Shivani767/llm-inferlite ...")
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/Shivani767/llm-inferlite.git",
        str(dest),
    ])
    REPO_ROOT = dest

BACKEND = REPO_ROOT / "backend"
assert (BACKEND / "research").exists(), (
    f"Could not find InferLite backend under {REPO_ROOT}. "
    "Run: !git clone --depth 1 https://github.com/Shivani767/llm-inferlite.git /content/llm-inferlite"
)
os.chdir(BACKEND)
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
print("REPO_ROOT", REPO_ROOT)
print("BACKEND ", BACKEND)

Repo not found. Cloning https://github.com/Shivani767/llm-inferlite ...
REPO_ROOT /content/llm-inferlite
BACKEND  /content/llm-inferlite/backend


In [ ]:
!pip install -q -r requirements.txt
!pip install -q -r requirements-colab.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.4 MB/s eta 0:00:00


In [ ]:
from research.capabilities import probe
from research.env import collect_environment
import json

env = collect_environment(seed=42)
caps = probe()
print("device:", caps["device"], "cuda:", caps["cuda"], "colab:", env.get("colab"))
print("gpu:", (env.get("torch") or {}).get("gpu"))
print("\nCapability matrix:")
for name, item in caps["experiments"].items():
    print(f"  [{'YES' if item['supported'] else 'NO ':3}] {name}: {item['reason']}")

## Run the Colab T4 suite

Default model: TinyLlama 1.1B. INT8/INT4 run if bitsandbytes + CUDA work. AWQ, GPTQ, GGUF, TensorRT-LLM, SmoothQuant, SqueezeLLM, and vLLM PagedAttention are attempted and recorded as **unsupported** when libraries or files are missing — they are never given fake TPS.

Speculative decoding uses TinyLlama as both target and draft so tokenizers match. That is a valid measurement of the algorithm, not a claim about a smaller draft model.

In [ ]:
from research.runner import load_config, run_config

cfg_path = REPO_ROOT / "configs" / "colab_t4.yaml"
cfg = load_config(cfg_path)
# Keep the first Colab run short enough for a free session. Increase later.
cfg["max_new_tokens"] = 24
cfg["measure_runs"] = 2
cfg["warmup_runs"] = 1
cfg["results_dir"] = str(BACKEND / "results" / "colab_t4")

summary = run_config(cfg, results_dir=cfg["results_dir"], make_plots=True)
print("measured", summary["n_measured"], "unsupported", summary["n_unsupported"], "error", summary["n_error"])
print("bundle", summary["bundle"])
print("csv", summary["csv"])
print("plots", summary["plots"])
print("pareto", json.dumps(summary["pareto"], indent=2, default=str)[:2000])

In [ ]:
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path(cfg["results_dir"]) / "figures"
if fig_dir.exists():
    for p in sorted(fig_dir.glob("*.png")):
        print(p.name)
        display(Image(filename=str(p)))
else:
    print("No figures yet. Check n_measured in the previous cell.")

## Optional: download a GGUF and measure llama.cpp

Skip this cell if `llama-cpp-python` is not installed. InferLite will not fabricate GGUF numbers.

In [ ]:
from research.engine import run_benchmark

try:
    import llama_cpp  # noqa: F401
    rec = run_benchmark(
        model_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
        method="gguf",
        backend="llama.cpp",
        gguf_file="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
        filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
        max_new_tokens=32,
        measure_runs=2,
        warmup_runs=1,
    )
    print(rec.status, rec.reason)
    print(rec.metrics.model_dump() if rec.metrics else None)
except Exception as exc:
    print("GGUF skipped:", type(exc).__name__, exc)

## Honesty checklist

- Do not copy old README tables that listed Llama-3-8B TensorRT-LLM TPS. Those were simulations.
- Cite only `status=measured` rows, with this notebook's environment dump.
- If a method is `unsupported`, report the reason, not a guessed speedup.